In [35]:
import pandas as pd
import numpy as np
import seaborn as sns

pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
pd.set_option('display.max_colwidth', None)

In [36]:
df0 = pd.read_csv(r'C:\Users\meet.gajera\OneDrive\Desktop\hcdr__\Data\cleaned\bureau_cleaned.csv')

In [37]:
df0.head()

,SK_ID_CURR,CREDIT_ACTIVE,DAYS_CREDIT,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_DEBT_MISSING,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_LIMIT_MISSING
0,215354,Closed,-497,91323.0,0.0,0,0.0,1
1,215354,Active,-208,225000.0,171342.0,0,0.0,1
2,215354,Active,-203,464323.5,0.0,1,0.0,1
3,215354,Active,-203,90000.0,0.0,1,0.0,1
4,215354,Active,-629,2700000.0,0.0,1,0.0,1


In [38]:
df = df0.copy()

In [39]:
df.shape

(1709712, 8)

In [40]:
df.columns

Index(['SK_ID_CURR', 'CREDIT_ACTIVE', 'DAYS_CREDIT', 'AMT_CREDIT_SUM',
       'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_DEBT_MISSING',
       'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_LIMIT_MISSING'],
      dtype='object')

In [41]:
# Convert days to years
df['YEARS_CREDIT'] = abs(df['DAYS_CREDIT']/365)

df.insert(
    df.columns.get_loc('DAYS_CREDIT'),
    'YEARS_CREDIT',
    df.pop('YEARS_CREDIT')
)

In [42]:
df.drop(columns=['DAYS_CREDIT'], inplace=True)

In [43]:
# Debt ratio
df['DEBT_RATIO'] = round(df['AMT_CREDIT_SUM_DEBT'] / (df['AMT_CREDIT_SUM']+1), 4)

- Shows how much of the credit is still unpaid.

In [44]:
# Credit utilization
df['CREDIT_UTILIZATION'] = round(df['AMT_CREDIT_SUM'] / (df['AMT_CREDIT_SUM_LIMIT']+1), 4)

In [45]:
# Has debt
df['HAS_DEBT'] = (df['AMT_CREDIT_SUM_DEBT']>0).astype(int)

In [46]:
df['IS_ACTIVE'] = (df['CREDIT_ACTIVE'] == 'Active').astype(int)
df['IS_CLOSED'] = (df['CREDIT_ACTIVE'] == 'Closed').astype(int)
df['IS_SOLD'] = (df['CREDIT_ACTIVE'] == 'Sold').astype(int)
df['IS_BAD_DEBT'] = (df['CREDIT_ACTIVE'] == 'Bad debt').astype(int)

In [47]:
df.drop(columns=['CREDIT_ACTIVE'], inplace=True)

In [48]:
df.columns

Index(['SK_ID_CURR', 'YEARS_CREDIT', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT',
       'AMT_CREDIT_SUM_DEBT_MISSING', 'AMT_CREDIT_SUM_LIMIT',
       'AMT_CREDIT_SUM_LIMIT_MISSING', 'DEBT_RATIO', 'CREDIT_UTILIZATION',
       'HAS_DEBT', 'IS_ACTIVE', 'IS_CLOSED', 'IS_SOLD', 'IS_BAD_DEBT'],
      dtype='object')

In [49]:
bureau_agg = df.groupby('SK_ID_CURR').agg({

    'AMT_CREDIT_SUM': ['sum', 'mean', 'max'],
    'AMT_CREDIT_SUM_DEBT': ['sum', 'mean'],
    'AMT_CREDIT_SUM_DEBT_MISSING': ['sum', 'mean'],
    'AMT_CREDIT_SUM_LIMIT': ['sum'],
    'AMT_CREDIT_SUM_LIMIT_MISSING': ['sum', 'mean'],

    'DEBT_RATIO': ['mean', 'max'],
    'CREDIT_UTILIZATION': ['mean'],

    'HAS_DEBT': ['sum'],
    'IS_ACTIVE': ['sum'],
    'IS_CLOSED': ['sum'],
    'IS_SOLD': ['sum'],
    'IS_BAD_DEBT': ['sum'],

    'YEARS_CREDIT': ['min', 'max', 'mean']

})

In [50]:
bureau_agg = bureau_agg.reset_index()

bureau_agg.columns = [
    '_'.join(col).strip('_') for col in bureau_agg.columns
]

In [51]:
bureau_agg.to_csv(r"C:\Users\meet.gajera\OneDrive\Desktop\hcdr__\Data\featured\bureau_featured.csv",index=False)